# Estimating behavioural equations

A ModelFlow model combines **identities**, which hold exactly, with **behavioural equations**, whose coefficients are estimated from data. Estimation is part of building the model: an equation tagged `<est=...>` is estimated while `Makemodel` — or the `%%Makemymodel` magic — processes it, and the estimated coefficients are baked into the `FRML` it emits.

Three estimation methods are built in and can be named in the tag with nothing imported: **`ols`**, ordinary least squares through `statsmodels`, for equations linear in their parameters; **`nls_lmfit`**, non-linear least squares through the `lmfit` package, which is also the only backend accepting coefficient constraints; and **`nls_eviews`**, for those who have EViews. All three take the same equation syntax, so the choice is only about what the equation looks like.

An estimator can also be built in Python — with `.with_defaults()` — when it needs options the tag cannot carry. That comes later in the chapter; nothing before it needs an import.

:::{note}
This chapter is the **syntax**, worked on small synthetic data where the right answer is known in advance. A complete estimation on real data — a model for Nepal, estimated equation by equation, assembled and given add factors until it reproduces history — is a supplement of its own: *ModelFlow - Estimation, a worked example*.
:::

In [1]:
import pandas as pd
import numpy as np

from modelclass import model
import modeljupytermagic                     # %%dataframe and %%Makemymodel magics
from modelconstruct_estimation import Makemodel

# Development support
%load_ext autoreload
%autoreload 2

## A first example, where the answer is known

Before any real data, a synthetic set where the right answer is known in advance. The data below comes from a consumption function with coefficients we choose ourselves:

$$ CON_t = 100 + 0.6\,Y_t + \varepsilon_t $$

$Y$ grows at 3 per cent a year and $\varepsilon$ is a relative disturbance with standard deviation `sd`. Estimating that same equation back out of the data shows what the estimator does — and turning `sd` down shows what the disturbance costs.

Nothing has to be imported to estimate. An equation tagged `<est=name>` is estimated while the model is built, and three method names are built in:

| Name | Backend | For |
|---|---|---|
| `ols` | `statsmodels` least squares | equations linear in their parameters |
| `nls_lmfit` | `lmfit` least squares | any equation; the only backend taking `ST.` constraints |
| `nls_eviews` | EViews | any equation, if EViews is installed |

The data, the sample and the coefficient names come from the `%%Makemymodel` line as `input_df=`, `smpl=` and `param_names=`.

In [2]:
def make_data(sd=0.02, years=range(2000, 2021), seed=42):
    '''Synthetic data from a known consumption function: CON = 100 + 0.6*Y

    sd is the standard deviation of the relative disturbance on CON.'''

    idx   = list(years)
    y     = pd.Series([1000 * 1.03**t for t in range(len(idx))], index=idx)
    con   = 100 + 0.6 * y
    noise = np.random.default_rng(seed).normal(0, sd, len(idx))
    return pd.DataFrame({'Y': y, 'CON': con * (1 + noise)})


small = make_data()
small.head()

,Y,CON
2000,1000.00000,704.266039
2001,1030.00000,703.065828
2002,1060.90000,747.594746
2003,1092.72700,769.850695
2004,1125.50881,745.052328


In [3]:
%%Makemymodel Mlinear input_df=small smpl=(2001,2020) param_names=['alfa','beta_longterm']
## A linear consumption function
The long-run relation, estimated by ordinary least squares. The true coefficients are 100 and 0.6:
><stoc,est=ols> con = alfa + beta_longterm*y

## A linear consumption function
The long-run relation, estimated by ordinary least squares. The true coefficients are 100 and 0.6:
```text
><stoc,est=ols> con = alfa + beta_longterm*y
```

**Estimation output**

| Item | Value |
|:--|:--|
| Estimator | ols |
| Requested sample | (np.int64(2001), np.int64(2020)) |
| Effective sample | (np.int64(2001), np.int64(2020)) |
| Original equation | CON = ALFA + BETA_LONGTERM*Y |
| Estimated equation | CON = (85.3489879901) + (0.6101288809)*Y |
| aic | 168.1593835 |
| bic | 170.150848 |
| R-squared | 0.9896705862 |
| Adj. R-squared | 0.9890967298 |

| Parameter | Estimate | t-stat |
|:--|--:|--:|
| ALFA__1 | 85.34898799 | 4.139 |
| BETA_LONGTERM__1 | 0.6101288809 | 41.528 |

The same relation can also be written as an error correction equation, which is the form a
behavioural equation usually takes in a macro model: a long-run share the variable returns
to, a short-run response to growth, and a speed $\lambda$ at which the gap closes. It is not
linear in its parameters, so it needs `nls_lmfit`.

Two things change with it. It goes in a model of its own, since two equations for `CON`
cannot live in the same one. And the sample cannot start in 2000: the equation refers to
`con(-1)` and `y(-1)`, which do not exist in the first row of the data, so the first year
pays for the lag. Every estimation output reports the sample it actually used next to the
one it was given.

In [4]:
%%Makemymodel Mecm input_df=small smpl=(2000,2020) param_names=['lambda','beta_shortterm','beta_longterm']
## Consumption as an error correction equation
><stoc,est=nls_lmfit> dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)

Item,Value
Estimator,nls_lmfit
Requested sample,"(np.int64(2000), np.int64(2020))"
Effective sample,"(np.int64(2001), np.int64(2020))"
Original equation,DLOG(CON) = -LAMBDA(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS(BETA_LONGTERM(1)))) + BETA_SHORTTERM(10)DLOG(Y)
Estimated equation,DLOG(CON) = -(0.6136030211)(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.6949833169)))) + (0.2955687108)DLOG(Y)
success,True
message,ftol termination condition is satisfied.
chisqr,0.008515199595
redchi,0.0005008940939
aic,-149.232696


Under each equation the magic prints what the estimation did: the effective sample, the equation before and after the coefficients were substituted, and each parameter with its t-statistic. `ALFA__1` and `BETA_LONGTERM__1` in the linear equation should land close to the 100 and 0.6 the data was built from — not on them, because the disturbance is in the way.

Writing `<est=ols>` on every equation gets tiring in a model of any size. A **model-wide default** is set with `estimator=` — either on the `%%Makemymodel` line or as a `Makemodel(...)` argument — and the equations then need only a bare `<est>`.

### What the noise costs

The disturbance is the only thing standing between the estimator and the true coefficients. Generating the same data with less and less of it shows how much it costs. `Makemodel` is used directly here rather than through the magic, because a magic is one cell and this wants a loop — the estimator object of each equation is kept in `estimation_records`:

In [5]:
def estimate(sd):
    '''Estimate con = alfa + beta_longterm*y on data disturbed by sd.

    estimator='ols' is the model-wide default, so the equation needs only <est>.
    param_names travels in estimator_kwargs, which is what the magic line's
    param_names= option fills in for you.
    Returns the estimator object of the one estimated equation.'''

    mm = Makemodel('><est> con = alfa + beta_longterm*y',
                   input_df=make_data(sd), smpl=(2000, 2020), estimator='ols',
                   estimator_kwargs={'param_names': ['alfa', 'beta_longterm']})
    return mm.estimation_records[0]['estimator_object']


noise_levels = [0.10, 0.05, 0.02, 0.01, 0.001]
ests = {sd: estimate(sd) for sd in noise_levels}

# true values: ALFA__1 = 100, BETA_LONGTERM__1 = 0.6
pd.DataFrame({sd: e.coef_ser for sd, e in ests.items()}).T.rename_axis('noise sd')

,ALFA__1,BETA_LONGTERM__1
noise sd,,
0.100,46.434862,0.637758
0.050,73.217431,0.618879
0.020,89.286972,0.607552
0.010,94.643486,0.603776
0.001,99.464349,0.600378


In [6]:
pd.DataFrame({sd: e.tvalues for sd, e in ests.items()}).T.rename_axis('noise sd')

,ALFA__1,BETA_LONGTERM__1
noise sd,,
0.100,0.493270,9.397386
0.050,1.555554,18.238412
0.020,4.742407,44.761490
0.010,10.053828,88.966621
0.001,105.659411,884.658965


As `sd` falls the coefficients close on 100 and 0.6 and the t-statistics grow. At the bottom of the table the equation is recovered almost exactly, which is the useful check: the estimator is not doing anything to the data beyond what the disturbance forces it to.

Outside a textbook the noise is not a knob. What the experiment shows is what to reach for when estimates come out imprecise, because everything that buys precision does it by improving the same ratio between signal and disturbance:

| | How it helps |
|---|---|
| A longer sample | More observations per parameter. `smpl=` is the first thing to widen — if the extra years are comparable |
| A better specification | A term that genuinely belongs moves variance out of the residual and into the equation |
| Fewer parameters | Each one is estimated from the same data; dropping one that the data cannot identify sharpens the rest |
| Constraints | Fixing or bounding a coefficient you already know spends information you have instead of asking the data for it — the `ST.` clause below |

The last one is the most useful when the sample is short, which in macro it usually is. The worked example in *ModelFlow - Estimation, a worked example* is estimated on nine annual observations, which is an ordinary length for a national accounts model.

## A named estimator with `.with_defaults()`

Everything so far has been supplied per cell — the data, the sample, the coefficient names, repeated on every magic line. When several equations share them, and especially when an estimation needs options a magic line cannot carry, build a **factory** once and refer to it by name.

`.with_defaults()` collects the options and returns a callable that constructs and runs an estimation for any equation it is given; any keyword passed to the callable, or any equation tag, overrides a stored default. This is also the one place where the estimator class itself has to be imported.

A factory in the namespace is used exactly like a built-in name — `<est=ls>` — since `Makemodel` resolves the tag against the calling namespace as well as the built-ins.

In [7]:
from modelestimator_new import Estimate_nls

ls = Estimate_nls.with_defaults(
    smpl            = (2000, 2020),
    input_df        = small,
    var_description = {'Y': 'Income', 'CON': 'Consumption'},
    param_names     = ['lambda', 'beta_shortterm', 'beta_longterm'],
)

est = ls("dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)",
         caption='Consumption, named coefficients')
est.coef_ser

BETA_LONGTERM__1      0.694983
BETA_SHORTTERM__10    0.295569
LAMBDA__1             0.613603
Name: Consumption, named coefficients, dtype: float64

This is the same equation as `Mecm` above, written with names instead of `c(1)`, `c(2)` and `c(10)`, so the estimates are the same and only the labels differ.

### Inspecting the result

Displayed on its own, the estimator object renders a full report — regression results and actual against fitted. It also exposes the result piece by piece:

| Attribute | Content |
|---|---|
| `coef_ser` | Estimated coefficients as a Pandas `Series` |
| `tvalues` | t-statistics as a `Series` (NaN for fixed or derived coefficients) |
| `org_eq_baked` | The equation with the estimated numbers substituted in place |
| `af_df` | Actual and fitted values over the estimation sample |
| `residuals_df` | Residuals over the estimation sample |
| `regression_model` | The native `lmfit` result object |
| `get_html_report()` | The full report as an HTML string |

`org_eq_baked` is the one to read first: it is the equation that will go into the model, and reading it back is the cheapest check that the equation estimated is the equation intended.

In [8]:
print(est.org_eq_baked)
est.tvalues

DLOG(CON) = -(0.6136030211)*(LOG(CON(-1))-LOG(Y(-1))-LOG(ABS((0.6949833169)))) + (0.2955687108)*DLOG(Y)


BETA_LONGTERM__1      2.439227e-06
BETA_SHORTTERM__10    3.473041e-08
LAMBDA__1             2.859759e+00
Name: Consumption, named coefficients, dtype: float64

## The full syntax of an estimation

Everything the estimator needs can be written into the equation string. The complete form is

```text
[<tags>] lhs = rhs [ ST. constraint ; constraint ; ... ]
```

The tags are read by `Makemodel` and `%%Makemymodel` — they are listed under *Estimation tags* below. The `ST.` clause is the subject of the next section. The rest is the equation itself.

Case is irrelevant: the whole string is upper-cased before parsing, so `dlog(con)` and `DLOG(CON)` are the same equation.

### Left hand side

The endogenous variable, optionally transformed — `con`, `log(con)`, `dlog(con)`, `diff(con)`.

With `ecm=True`, the default for the NLS backends, the left hand side **must** be `dlog(var)`: the equation is read as an error-correction form, and `c(2)` — if it is present — is treated as the speed of adjustment. For a plain level or log equation use `ols`, which sets `ecm=False`. That is why `con = alfa + beta_longterm*y` is accepted above and would be rejected by `nls_lmfit`.

### Coefficients

A coefficient is a prefix and a number:

| Written | Becomes | Where the prefix comes from |
|---|---|---|
| `c(1)`, `c(2)` | `C__1`, `C__2` | `est_param`, which defaults to `'C'` |
| `lambda(1)`, `beta_shortterm(10)` | `LAMBDA__1`, `BETA_SHORTTERM__10` | a name declared in `param_names` |
| `lambda` | `LAMBDA__1` | a bare name is number 1 of that prefix |

Named prefixes are worth the small effort: a coefficient called `LAMBDA` is checked against economic sense more easily than one called `C__2`. Declare them with `param_names=` — on the `%%Makemymodel` line, as a `Makemodel(..., estimator_kwargs={'param_names': [...]})` argument, or stored once in a `.with_defaults()` factory.

The numbers are labels, not positions — `beta_shortterm(10)` beside `beta_longterm(1)` is perfectly ordinary, and the gaps are what make the numbering useful for keeping a coefficient's identity stable while an equation is being reworked. Setting `est_param='B'` renames the anonymous prefix, and then both `C(1)` and `B(1)` mean `B__1`.

This is also the one syntactic trap in the language. A name with a number after it is a coefficient **only** if its prefix was declared; otherwise `alfa(1)` reads as a *lead* of a variable called `ALFA`. The check below catches it, but the error talks about an unknown variable rather than a missing `param_names` entry.

### Right hand side

Ordinary ModelFlow expressions: arithmetic, lags such as `y(-1)`, and functions — `log`, `dlog`, `diff`, `exp`, `abs`. Each side must parse as Python, and a missing operator is reported with its position.

Every name which is not a declared coefficient has to be a column in `input_df`. An unknown one raises rather than entering the estimation as an all-NaN column, which is worth having: silently fitting against a column of NaN is the failure that produces a plausible-looking, meaningless coefficient vector.

### Which backend does what

| | ordinary least squares | non-linear least squares | EViews |
|---|---|---|---|
| Written in a tag as | `<est=ols>` | `<est=nls_lmfit>` | `<est=nls_eviews>` |
| Class | `Estimate_ols` | `Estimate_nls_lmfit` | `Estimate_nls_eviews` |
| Engine | statsmodels | lmfit | EViews |
| Equation | linear in the parameters | any | any |
| `ecm` default | `False` | `True` | `True` |
| `ST.` constraints | not supported | supported | not supported |
| Intercept | write it explicitly, as `alfa` or `c(1)`; omitting it fits through the origin and warns | as written | as written |

`Estimate_nls` is a factory over the last two: it never produces an instance of itself, but returns the backend named by `solver='lmfit'` or `solver='eviews'`. All of the classes carry `.with_defaults()`, and a factory built from any of them can be named in a tag just like a built-in.

## Constraints — the `ST.` clause

Coefficients can be constrained directly in the equation string after the keyword `ST.`, with individual constraints separated by `;`:

    y = c(1) + c(2)*x + alfa*z st. c(1)~0.5; c(2)>0; alfa:=c(1)+c(2)

| Form | Meaning |
|---|---|
| `NAME = value` | Fix the coefficient at `value` (not estimated) |
| `NAME ~ value` | Starting value (still estimated) |
| `NAME > x` / `NAME >= x` | Lower bound |
| `NAME < x` / `NAME <= x` | Upper bound |
| `NAME = [lo, hi]` | Lower and upper bound |
| `NAME := expr` | Derived coefficient: an algebraic function of other coefficients |

The same constraints can be supplied programmatically through the `constraints=` option as per-parameter `lmfit` settings, e.g. `constraints={'LAMBDA__1': {'value': 0.3, 'vary': False}}`.

Below the speed of adjustment $\lambda$ is fixed at 0.3, so only the two $\beta$s are estimated:

In [9]:
est_fix = ls("dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y) st. lambda = 0.3",
             caption='Consumption, lambda fixed')
est_fix.coef_ser

BETA_LONGTERM__1      0.884157
BETA_SHORTTERM__10   -1.853228
LAMBDA__1             0.300000
Name: Consumption, lambda fixed, dtype: float64

And here $\lambda$ is restricted to the interval $[0.1,\;0.5]$ while the short-run elasticity is kept non-negative:

In [10]:
est_bound = ls("dlog(con) = -lambda*(log(con(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y) st. lambda = [0.1, 0.5]; beta_shortterm(10) >= 0",
               caption='Consumption, bounded')
est_bound.coef_ser

BETA_LONGTERM__1      0.401768
BETA_SHORTTERM__10    9.672186
LAMBDA__1             0.500000
Name: Consumption, bounded, dtype: float64

## Options reference

All options accepted by `Estimate_nls` — directly or through `.with_defaults()`:

| Option | Default | Description |
|---|---|---|
| `org_eq` | — | The equation (positional when calling the factory) |
| `input_df` | `None` | `DataFrame` with the data; time on the index |
| `smpl` | `(2002, 2018)` | Estimation sample, inclusive |
| `est_param` | `'C'` | Prefix for numbered coefficients `C(1)`, `C(2)`, ... |
| `param_names` | `None` | Additional named coefficient prefixes, e.g. `['lambda', 'beta_longterm']` |
| `caption` | `'Estimation of '` | Caption used in reports |
| `var_description` | `{}` | `{variable: description}` used to label output |
| `constraints` | `{}` | Per-parameter lmfit settings, e.g. `{'C__1': {'min': 0, 'max': 1}}`; also filled from an inline `ST.` clause |
| `default_params` | `{}` | Per-parameter starting values and bounds, e.g. `{'C__2': {'value': -0.1, 'min': -1.0, 'max': 0.0}}` |
| `method` | `'least_squares'` | Minimizer method passed to `lmfit.minimize` |
| `fit_kws` | `{}` | Extra keyword arguments forwarded to `lmfit.minimize` |
| `coef_dict` | `{}` | Numeric substitutions applied to placeholders before parsing |
| `frml_name` | `'<STOC,DAMP>'` | FRML tag used when the estimated equation is emitted |
| `add_add_factor` | `True` | Generate an add factor in the normalized equation |
| `make_fixable` | `True` | Generate `_X`/`_D` fixing variables |
| `make_fitted` | `False` | Generate a `_FITTED` equation |

Every parameter without an explicit starting value starts at `0.1`. For ECM equations the speed-of-adjustment often needs help: give it a starting value or bounds through `default_params` or an `ST.` clause.

## Estimation while building a model

Estimation really shines inside `Makemodel`: an equation tagged with `<estimator=...>` is estimated during model construction, and the estimated coefficients are baked into the emitted `FRML`. The name is resolved first against an explicit `estimator_classes=` mapping, then against the calling namespace — which is how a factory such as `ls` above is found — and last against the built-in methods `ols`, `nls_lmfit` and `nls_eviews`. A variable in the notebook therefore shadows a built-in of the same name.

### Estimation tags

| Tag | Effect |
|---|---|
| `<estimator=name>` (alias `<est=name>`) | Estimate this equation with the estimator `name` — a factory in the namespace or a built-in method |
| `<estimator>` / `<est>` | Estimate with the model-wide default passed as `Makemodel(..., estimator=...)` or `estimator=` on the magic line |
| `<stoc>` | Mark the equation as behavioural: implies `<exo>` (fixable) and `<add>` (add factor) |
| `<smpl=start end>` | Equation-local estimation sample; overrides the estimator and `Makemodel` defaults |
| `<caption='text'>` | Caption used for the equation in estimation output and reports |
| `<constraints=...>` | Coefficient constraints, appended to the equation as an `ST.` clause |
| `<drop>` | Skip the equation when the FRMLs are emitted (the text is kept in the documentation) |

An equation tagged `<stoc>` comes back from construction **normalized** — solved for its endogenous variable rather than for a log difference — with an add factor and a fixing pair attached, and with a `<CALC_ADD_FACTOR>` equation generated alongside it. *ModelFlow - Estimation, a worked example* builds a complete model this way and reads the emitted `FRML`s.

### Estimation output and reports

A `Makemodel` built from estimated equations keeps the estimations, not only their results:

- `.render_est` renders the model text with estimation tables (coefficients, t-statistics, fit measures) — the `%%Makemymodel` magic does this automatically.
- `.tvalues` collects the t-statistics of all estimated equations in one `DataFrame`.
- `.estimation_report(open_file=True)` writes an HTML report with actual-versus-fitted plots for every estimated equation; `report_all=True` also includes the identities.
- `.report().show()` displays the full model documentation — prose, equations and collapsible estimation panels.
- `.estimation_records` holds the raw record of every estimation run during construction, including the estimator object itself; `.showestimation_records` prints it.

## Quick reference

```text
[<tags>] lhs = rhs [ ST. constraint ; constraint ; ... ]     # the full equation form
```

Nothing imported — the built-in method names, with the data, sample and coefficient names on the magic line:

```text
%%Makemymodel Mdemo input_df=df smpl=(2000,2020) estimator=ols param_names=['alfa','beta_longterm']
## Consumption
><stoc,est> con = alfa + beta_longterm*y
## Investment
><stoc,est=nls_lmfit> dlog(inv) = -lambda*(log(inv(-1))-log(y(-1))-log(abs(beta_longterm(1)))) + beta_shortterm(10)*dlog(y)
```

A factory, when several equations share the same options:

```python
from modelestimator_new import Estimate_nls

ls = Estimate_nls.with_defaults(smpl=(start, end), input_df=df,
                                param_names=['lambda', 'beta_shortterm', 'beta_longterm'],
                                var_description={...})

est = ls("dlog(x) = -lambda*(log(x(-1))-log(y(-1))-log(abs(beta_longterm(1))))"
         " + beta_shortterm(10)*dlog(y) st. lambda=[0.1, 0.5]")
est.coef_ser; est.tvalues; est.org_eq_baked
```

Building and using the model:

```python
mm = Makemodel("><stoc,est=ls,caption='...',smpl=2012 2019> dlog(x) = ...")
mm.render_est; mm.tvalues; mm.estimation_report(open_file=True); mm.report().show()
mm.estimation_records[0]['estimator_object']        # the estimator of one equation
solvable = mm.mmodel
baseline = mm.init_addfactors(df, start, end, check=True)
```